# 06. Heterogeneity and uplift funnel

This section is a post-confirmatory exploratory extension. S6 already closed the confirmatory sealed-test evaluation: the pre-selected UpliftTree did not show confirmatory advantage over the response-targeting baseline (Delta Qini = -0.0088; 95% CI [-0.0492, +0.0302]). Therefore, this notebook does not reopen the sealed test, retrospectively re-select a model, or change the main project conclusion.

The goal is operational: understand which profiles receive higher estimated uplift scores in development and check whether the ranking optimized for `visit` carries the same ordering for `conversion` and `spend`.

## Contents

- [7.1 Allowed data](#s7-1)
- [7.2 Exploratory development ranking](#s7-2)
- [7.3 Top vs. bottom ranking profile](#s7-3)
- [7.4 Observed outcomes by quantile](#s7-4)
- [7.5 Funnel uplift](#s7-5)
- [7.6 Agreement between rankings](#s7-6)
- [7.7 Interpretable surrogate](#s7-7)
- [7.8 S7 artifacts](#s7-8)
- [7.9 Sleeping Dogs and final reading](#s7-9)

---

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / 'src').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import POOLED_TREATMENT_COL, PROJECT_ROOT as CONFIG_PROJECT_ROOT, SEED
from src.data import add_pooled_treatment, load_hillstrom
from src.i18n import make_lang
from src.reports import build_s7_heterogeneity_report
from src.splits import get_train_val

PROJECT_ROOT = CONFIG_PROJECT_ROOT
lang = make_lang('en')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
ARTIFACTS_S7_DIR = PROJECT_ROOT / 'artifacts' / 's7'
ARTIFACTS_S7_DIR.mkdir(parents=True, exist_ok=True)

Failed to import duecredit due to No module named 'duecredit'


<a id="s7-1"></a>

## 7.1 Allowed data

We use only train and validation materialized by the persisted manifests. The output below is a simple check: the sealed test is not loaded in this notebook. The `conversion` count in the full dataset is used only to size the rarity of the outcome, without opening sealed-holdout outcomes or covariates through `load_sealed_test`.

In [2]:
df = load_hillstrom()
df_pooled = add_pooled_treatment(df)
train_df, val_df = get_train_val(df_pooled)

print(f'train={len(train_df):,} | validation={len(val_df):,} | sealed test not loaded')
print(f"conversion positives in full dataset={int(df['conversion'].sum()):,} / {len(df):,} ({df['conversion'].mean():.2%})")

train=38,400 | validation=12,800 | sealed test not loaded
conversion positives in full dataset=578 / 64,000 (0.90%)


<a id="s7-2"></a>

## 7.2 Exploratory development ranking

To keep this step cheap and interpretable, we fit an X-learner with a shallow tree (`max_depth=4`) on train and score validation. Because the pooled experiment is randomized, with two treated arms and one control arm, the known propensity is P(T=1|X)=2/3 for every row. This value follows from the experimental design rather than tuning; passing it explicitly prevents `causalml` from fitting a nuisance propensity model. The same specification is repeated for `visit`, `conversion`, and `spend`. This ranking does not replace the historical primary model from S6; it is only a descriptive tool for profiles, quantiles, and funnel diagnostics.

In [3]:
report = build_s7_heterogeneity_report(train_df, val_df, full_df=df)
profile = report['profile']
quantile_outcomes = report['quantile_outcomes']
funnel_metrics = report['funnel_metrics']
funnel_spearman = report['funnel_spearman']
surrogate = report['surrogate']

print(f"conversion positives={report['conversion_positives']:,} | rate={report['conversion_rate']:.2%}")

conversion positives=578 | rate=0.90%


<a id="s7-3"></a>

## 7.3 Top vs. bottom ranking profile

The table compares the lowest and highest quantiles of the estimated uplift score for `visit`. Continuous variables appear as means; categorical and binary variables appear as the share of each level at both extremes. This describes the estimated high-uplift group, not individually observable Persuadables, and it does not formally confirm causal heterogeneity.

In [4]:
profile.head(16).round(4)

,variable,level,bottom_quantile,top_quantile,delta_top_minus_bottom
0,history,mean,237.8625,378.4234,140.5610
1,recency,mean,6.8266,4.3777,-2.4488
2,womens,0,0.8535,0.0039,-0.8496
3,womens,1,0.1465,0.9961,0.8496
4,mens,0,0.0586,0.5887,0.5301
5,mens,1,0.9414,0.4113,-0.5301
6,channel,Web,0.7023,0.3883,-0.3141
7,history_segment,1) $0 - $100,0.4234,0.1559,-0.2676
8,zip_code,Rural,0.4348,0.1895,-0.2453
9,channel,Phone,0.2129,0.3965,0.1836


<a id="s7-4"></a>

## 7.4 Observed outcomes by quantile

This table organizes validation by quantiles of the `visit` score and shows observed means for `visit`, `conversion`, and `spend`. The question is whether the top of the funnel moves together with outcomes closer to revenue. If the ordering improves `visit` but not `spend`, the operational message is clear: optimizing for visit may not optimize for revenue.

In [5]:
quantile_outcomes.round(4)

,quantile,n,score_visit_mean,visit_mean,conversion_mean,spend_mean
0,1,2560,0.0165,0.1672,0.0090,1.1366
1,2,2560,0.0487,0.1305,0.0074,1.2344
2,3,2560,0.0634,0.1070,0.0070,0.7586
3,4,2560,0.0764,0.1375,0.0043,0.3075
4,5,2560,0.0959,0.1918,0.0164,2.0993


<a id="s7-5"></a>

## 7.5 Funnel uplift

Now each outcome receives its own exploratory ranking. `conversion` is especially difficult: there are only 578 positives in 64,000 rows, about 0.9% of the dataset. For this reason, apparent bottom-funnel differences should be read carefully; divergent rankings suggest the top-of-funnel proxy can mislead, but the sample may not be large enough to quantify by how much with precision.

In [6]:
funnel_metrics.round(4)

,outcome,qini_auc,uplift_auc,uplift_at_30pct,incremental_mean_top_30pct
0,visit,0.0615,0.0365,0.1024,0.0930
1,conversion,0.0216,0.0011,0.0039,0.0064
2,spend,NaN,NaN,NaN,0.8878


<a id="s7-6"></a>

## 7.6 Agreement between rankings

The Spearman matrix compares the ordering induced by the `visit`, `conversion`, and `spend` scores. Low or unstable correlations reinforce the funnel thesis: the most promising customer for a visit does not have to be the same customer for purchase or spend.

In [7]:
funnel_spearman.pivot(index='score_a', columns='score_b', values='spearman_corr').round(3)

score_b,conversion,spend,visit
score_a,,,
conversion,1.000,0.131,0.652
spend,0.131,1.000,0.094
visit,0.652,0.094,1.000


<a id="s7-7"></a>

## 7.7 Interpretable surrogate

The tree below is an exploratory surrogate trained to approximate membership in the top quantile of the `visit` score. The target here is not the causal outcome; it is the estimated high-ranking label. Therefore, the rules are a compact description of the ranking, not a confirmatory causal explanation.

In [8]:
print(f"positive_rate={surrogate['positive_rate']:.2%}")
print(f"balanced_accuracy={surrogate['balanced_accuracy']:.3f}")
print(surrogate['rules'])

positive_rate=20.00%
balanced_accuracy=0.835
|--- num__womens <= 0.50
|   |--- num__history <= 693.66
|   |   |--- class: 0
|   |--- num__history >  693.66
|   |   |--- class: 0
|--- num__womens >  0.50
|   |--- num__mens <= 0.50
|   |   |--- num__recency <= 3.50
|   |   |   |--- class: 1
|   |   |--- num__recency >  3.50
|   |   |   |--- class: 0
|   |--- num__mens >  0.50
|   |   |--- num__history <= 802.15
|   |   |   |--- class: 1
|   |   |--- num__history >  802.15
|   |   |   |--- class: 1



<a id="s7-8"></a>

## 7.8 S7 artifacts

We persist only tables derived from train/validation for review and publication. Nothing in `artifacts/s6/` is read to recalculate results, changed, or overwritten.

In [9]:
profile.to_csv(ARTIFACTS_S7_DIR / 's7_top_bottom_profile.csv', index=False)
quantile_outcomes.to_csv(ARTIFACTS_S7_DIR / 's7_quantile_outcomes.csv', index=False)
funnel_metrics.to_csv(ARTIFACTS_S7_DIR / 's7_funnel_metrics.csv', index=False)
funnel_spearman.to_csv(ARTIFACTS_S7_DIR / 's7_funnel_spearman.csv', index=False)
print(f'S7 artifacts saved to {ARTIFACTS_S7_DIR.relative_to(PROJECT_ROOT)}')

S7 artifacts saved to artifacts\s7


<a id="s7-9"></a>

## 7.9 Sleeping Dogs and final reading

The dataset has no `unsubscribe`, opt-out, complaint, or any direct harm measure. Therefore, negative uplift in `visit` is only a weak signal of possible adverse response, not evidence of Sleeping Dogs in a strong operational sense.

The correct S7 conclusion is limited: exploratory profiles and rankings help think about segmentation and the risk of using `visit` as a revenue proxy, but S6 remains negative for the primary confirmatory hypothesis. S7 is not a back door for declaring that uplift modeling beat response targeting.